# Exercise 43 — `raise`: when and why

## Concept

`raise` signals that the function **cannot complete its contract** — the inputs are wrong, an external system gave bad data, or an invariant was violated.

```python
def divide(a, b):
    if b == 0:
        raise ValueError("b must be non-zero")
    return a / b
```

### Pick the right built-in exception

- **`ValueError`** — argument has the right type but a bad value (`int("abc")`, negative count for a count, etc.)
- **`TypeError`** — argument has the wrong type
- **`KeyError`** / **`IndexError`** — missing key / out-of-range index in a container you own
- **`FileNotFoundError`** — file you expected isn't there
- **`RuntimeError`** — generic; use sparingly when nothing more specific fits

Don't invent your own exception types until built-ins genuinely don't fit. Custom exceptions are a Module 2+ topic.

### Re-raise after logging

When you catch an exception but can't actually fix the situation, log it and re-raise with bare `raise` — preserves the traceback:

```python
try:
    risky()
except Exception:
    log_failure(...)
    raise          # not `raise e` — bare `raise` keeps the original traceback
```

### Don't catch what you raise

If a function raises `ValueError` for bad input, that's a **caller contract**. The caller decides whether to catch it. Don't wrap your own `raise` in `try`/`except`.

## Task 1 — Validate and raise

Write `divide_safe(a, b)` that returns `a / b` but raises `ValueError("b must be non-zero")` when `b == 0`.

```python
def divide_safe(a: float, b: float) -> float:
    ...
```

Don't catch the exception — just `raise` and let it propagate.

In [ ]:
def divide_safe(a: float, b: float) -> float:
    pass  # TODO

assert divide_safe(10, 2) == 5

try:
    divide_safe(1, 0)
except ValueError as e:
    assert str(e) == "b must be non-zero"
else:
    raise AssertionError("expected ValueError, got nothing")
print("ok")


## Task 2 — Pick the right exception type

Write `get_field(record, field)` that returns `record[field]`, with these rules:
- If `record` is not a `dict` → `TypeError("record must be a dict")`
- If `field` is not in `record` → `KeyError(field)` (yes, just the field name — that's the convention for `KeyError`)
- Otherwise → return the value

```python
def get_field(record, field):
    ...
```

This is a small but real example of using **built-in types correctly** so callers can write `except KeyError` to handle missing fields specifically.

In [ ]:
def get_field(record, field):
    pass  # TODO

assert get_field({"a": 1}, "a") == 1

try:
    get_field("not a dict", "a")
except TypeError as e:
    assert "dict" in str(e)
else:
    raise AssertionError("expected TypeError")

try:
    get_field({"a": 1}, "b")
except KeyError as e:
    assert e.args == ("b",)
else:
    raise AssertionError("expected KeyError")
print("ok")


## Task 3 — Log and re-raise

Write a helper that runs a callable, logs any exception to a provided list, and **re-raises**. The point is to add context (in this case, recording the failure) without swallowing the error.

```python
def run_and_log(fn, log: list[str]):
    """Call fn(). On exception, append str(exc) to `log` and re-raise.
    On success, return fn()'s return value.
    """
```

Use a bare `raise` to preserve the traceback.

The test calls it twice: once happy path, once with a failing function.

In [ ]:
def run_and_log(fn, log: list[str]):
    pass  # TODO

log: list[str] = []
assert run_and_log(lambda: 42, log) == 42
assert log == []  # nothing logged on success

def boom():
    raise RuntimeError("explosion")

try:
    run_and_log(boom, log)
except RuntimeError as e:
    assert str(e) == "explosion"
else:
    raise AssertionError("expected RuntimeError to propagate")

assert log == ["explosion"]
print("ok")
